##Task board MCP + planner agent

This notebook demonstrates a **custom stdio MCP server** ([`task_board_server.py`](task_board_server.py)) that persists tasks to `tasks.json`, wired into the **OpenAI Agents SDK** with `MCPServerStdio`.

The agent uses the task-board tools to capture a plan; it uses **`mcp-server-fetch`** to read public URLs and turn that content into actionable tasks.

In [10]:
import os
import shutil
import sys
from contextlib import AsyncExitStack
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

True

In [ ]:
def find_six_mcp() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "community_contributions" / "task_board_server.py").is_file():
        return cwd
    if (cwd / "task_board_server.py").is_file():
        return cwd.parent
    for p in [cwd, *cwd.parents]:
        if (p / "community_contributions" / "task_board_server.py").is_file():
            return p
    raise FileNotFoundError(
        "Could not locate 6_mcp (missing community_contributions/task_board_server.py). "
        "cd into agents/6_mcp and restart the kernel, or open the notebook from that folder."
    )


SIX_MCP = find_six_mcp()
os.chdir(SIX_MCP)
print("cwd:", Path.cwd())

TASKS_JSON = SIX_MCP / "community_contributions" / "tasks.json"
os.environ.setdefault("TASKS_JSON_PATH", str(TASKS_JSON))

In [ ]:


USE_FETCH_MCP = True

TASK_SERVER_SCRIPT = SIX_MCP / "community_contributions" / "task_board_server.py"
REL_TASK_SERVER = "community_contributions/task_board_server.py"

_uv = shutil.which("uv")
if _uv:
    task_board_mcp = {"command": _uv, "args": ["run", REL_TASK_SERVER]}
else:
    task_board_mcp = {"command": sys.executable, "args": [str(TASK_SERVER_SCRIPT)]}
    print(
        "Note: `uv` not on PATH; spawning task board with this kernel's Python:",
        sys.executable,
    )

fetch_mcp = None
if USE_FETCH_MCP:
    _uvx = shutil.which("uvx")
    _fetch = shutil.which("mcp-server-fetch")
    if _uvx:
        fetch_mcp = {"command": _uvx, "args": ["mcp-server-fetch"]}
    elif _fetch:
        fetch_mcp = {"command": _fetch, "args": []}
    else:
        print(
            "Warning: `uvx` and `mcp-server-fetch` not on PATH; disabling fetch MCP. "
            "Set USE_FETCH_MCP = False or install so `mcp-server-fetch` is available."
        )
        USE_FETCH_MCP = False

mcp_server_params = [task_board_mcp] + ([fetch_mcp] if USE_FETCH_MCP and fetch_mcp else [])

In [ ]:
async def print_mcp_tool_counts():
    async with AsyncExitStack() as stack:
        servers = [
            await stack.enter_async_context(
                MCPServerStdio(params, client_session_timeout_seconds=60)
            )
            for params in mcp_server_params
        ]
        for i, s in enumerate(servers):
            await s.connect()
            tools = await s.list_tools()
            print(f"mcp_server[{i}] -> {len(tools)} tools")
            for t in tools:
                print("  -", t.name)


await print_mcp_tool_counts()

In [ ]:
USER_PROMPT = """Plan work for a small open-source contribution:
1) Add a short README section documenting this task-board MCP.
2) Open a PR with the notebook and server.
Break this into concrete tasks using the task board tools (titles + optional descriptions + priorities).
Then call list_tasks to show open items and finish with a brief summary in markdown."""

planner_instructions = """You are a planning agent with access to a task board via MCP tools.
You must use the task board tools to record work: add_task, list_tasks, complete_task, set_priority as needed.
Do not invent task ids; use ids returned by add_task.
If fetch/web tools are available, you may fetch public HTTPS pages the user mentions to extract actionable items—stay within reasonable rate limits and prefer official docs.
Keep the final reply concise and include the current open task list (titles and ids)."""


async def run_planner():
    async with AsyncExitStack() as stack:
        servers = [
            await stack.enter_async_context(
                MCPServerStdio(params, client_session_timeout_seconds=90)
            )
            for params in mcp_server_params
        ]
        for s in servers:
            await s.connect()
        agent = Agent(
            name="TaskPlanner",
            instructions=planner_instructions,
            model="gpt-4o-mini",
            mcp_servers=servers,
        )
        with trace("TaskBoardDemo"):
            return await Runner.run(agent, USER_PROMPT, max_turns=25)


result = await run_planner()
display(Markdown(result.final_output))

In [ ]:
path = Path(os.environ["TASKS_JSON_PATH"])
if path.is_file():
    display(Markdown("```json\n" + path.read_text(encoding="utf-8") + "\n```"))
else:
    print("No tasks file yet (agent may not have called add_task):", path)